# RAG Pipeline Test Playground
Use this notebook to run and test individual components of the RAG pipeline.

## 0. Setup

In [1]:
import os, sys
from pathlib import Path

# The 'rag' directory is itself the package, so its parent must be on sys.path
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in a .env file at the project root"
print("API key loaded.")

API key loaded.


## 1. Test: Parser
Parse a local file and inspect the raw parent chunks returned.

In [2]:
FILE_PATH = "dynamo.pdf"   # ← change to your file
PARSER    = "unstructured"   # "unstructured" | "docling"

In [3]:
source = Path(FILE_PATH).name

In [4]:
from unstructured.partition.auto import partition
elements = partition(
            filename=source, # Path to your PDF file
            strategy="hi_res", # Use the processing method of extraction
            extract_image_block_to_payload=True, # Store images as base64 data you can actually use
            infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
            extract_image_block_types=["Image"], # Grab images found in the PDF
        )

/Users/aditya.bansal@zomato.com/projects/rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 9368.15it/s]


In [5]:
from rag.parsers.unstructured_parser import UnstructuredParser
from rag.parsers.docling_parser import DoclingParser

parser = UnstructuredParser() if PARSER == "unstructured" else DoclingParser()
raw_parents = parser.parse(FILE_PATH)

print(f"{len(raw_parents)} parent sections found")

No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 10598.02it/s]


40 parent sections found


In [6]:
from collections import Counter

for index, parent in enumerate(raw_parents):
    counts = Counter(child.chunk_type.value for child in parent.children)
    counts_str = "  ".join(f"{k}={v}" for k, v in sorted(counts.items()))
    print(f"Index:{index}  [{parent.metadata.get('section_title', '')[:50]}]  total={len(parent.children)}  {counts_str}")

Index:0  [Dynamo: Amazon’s Highly Available Key-value Store]  total=1  text=1
Index:1  [Amazon.com]  total=0  
Index:2  [ABSTRACT]  total=4  text=4
Index:3  [Categories and Subject Descriptors]  total=1  text=1
Index:4  [General Terms]  total=1  text=1
Index:5  [1. INTRODUCTION]  total=9  text=9
Index:6  [2. BACKGROUND]  total=3  text=3
Index:7  [2.1 System Assumptions and Requirements]  total=5  text=5
Index:8  [2.2 Service Level Agreements (SLA)]  total=2  image=1  text=1
Index:9  [Figure 1: Service-oriented architecture of Amazon’]  total=7  text=7
Index:10  [2.3 Design Considerations]  total=9  text=9
Index:11  [3. RELATED WORK]  total=0  
Index:12  [3.1 Peer to Peer Systems]  total=3  text=3
Index:13  [3.2 Distributed File Systems and Databases]  total=6  image=1  text=5
Index:14  [3.3 Discussion]  total=1  text=1
Index:15  [4. SYSTEM ARCHITECTURE]  total=4  table=1  text=3
Index:16  [4.1 System Interface]  total=2  text=2
Index:17  [4.2 Partitioning Algorithm]  total=7  text=7
In

In [7]:
print(raw_parents[15].children[2].raw_content)

<table><thead><tr><th>Problem</th><th></th><th>Advantage</th></tr></thead><tbody><tr><td>Partitioning</td><td>Consistent Hashing</td><td>Incremental Scalability</td></tr><tr><td>High Availability for writes</td><td>Vector clocks with reconciliation during reads</td><td>Version size is decoupled from update rates.</td></tr><tr><td>Handling temporary failures</td><td>Sloppy Quorum and hinted handoff</td><td>Provides high availability and durability guarantee when some of the replicas are not available.</td></tr><tr><td>Recovering from permanent failures</td><td>Anti-entropy using Merkle trees</td><td>Synchronizes divergent replicas in the background.</td></tr><tr><td>Membership and failure detection</td><td>Gossip-based membership protocol and failure detection.</td><td>Preserves symmetry and avoids having a centralized registry for storing membership and node liveness information.</td></tr></tbody></table>


## 2. Test: Chunker
Merge / split parent sections into child chunks.

In [8]:
from collections import Counter
from rag.chunkers.chunker import process_all_parents

def chunk_summary(chunks):
    rows = []
    for parent in chunks:
        counts = Counter(child.chunk_type.value for child in parent.children)
        rows.append((parent.id, parent.metadata.get("section_title", "")[:50], len(parent.children), counts))
    return rows

before = chunk_summary(raw_parents)

parent_chunks = process_all_parents(raw_parents)

after = chunk_summary(parent_chunks)

print(f"{'ID':34} {'Section':52} {'before':>6}  {'after':>5}  {'before_types':30}  after_types")
print("-" * 160)
for (pid, title, b_total, b_counts), (_, _, a_total, a_counts) in zip(before, after):
    b_str = "  ".join(f"{k}={v}" for k, v in sorted(b_counts.items()))
    a_str = "  ".join(f"{k}={v}" for k, v in sorted(a_counts.items()))
    print(f"{pid:34} {title:52} {b_total:>6}  {a_total:>5}  {b_str:30}  {a_str}")

ID                                 Section                                              before  after  before_types                    after_types
----------------------------------------------------------------------------------------------------------------------------------------------------------------
589eed1596cec2430f2a12bd052d78dd   Dynamo: Amazon’s Highly Available Key-value Store         1      1  text=1                          text=1
ec0f1a0157b4aa384c7d7f5d3b7711c4   Amazon.com                                                0      0                                  
d8f620b40c5f106bf9eee0934a87d161   ABSTRACT                                                  4      1  text=4                          text=1
f8a5b162a6ae572744c2b41198ee9f96   Categories and Subject Descriptors                        1      1  text=1                          text=1
5f7aacd8c4dda68fe56fce63eb9fe77e   General Terms                                             1      1  text=1                     

## 3. Test: LLM Enricher
Enrich non-text chunks (tables, images) using an LLM.

In [ ]:
from rag.enrichers.enricher import LLMEnricher, build_embedding_content
from rag.models import ChunkType

# Enrich a small slice to limit API cost during testing
SAMPLE_SIZE = 3

enricher = LLMEnricher(model="gpt-4o", max_concurrency=5)

sample_parents = [
    p for p in parent_chunks
    if any(child.chunk_type in (ChunkType.IMAGE, ChunkType.TABLE) for child in p.children)
][:SAMPLE_SIZE]

print(f"Selected {len(sample_parents)} parents with image/table children")

await enricher.enrich_all(sample_parents)
build_embedding_content(sample_parents)

Selected 3 parents with image/table children

────────────────────────────────────────────────────────────
  BEFORE ENRICHMENT
────────────────────────────────────────────────────────────
  ID                                 Section                                  total  types
  ────────────────────────────────── ──────────────────────────────────────── ─────  ──────────────────────────────
  07e9f46ae827ddd7363c8768764613bc   2.2 Service Level Agreements (SLA)           2  image=1  text=1
  c5eba510fc88ca3a5fdeadda8cb79739   3.2 Distributed File Systems and Databas     3  image=1  text=2
  e10d4d56d79188abf6b439393a00fb5d   4. SYSTEM ARCHITECTURE                       3  table=1  text=2
────────────────────────────────────────────────────────────


────────────────────────────────────────────────────────────
  AFTER ENRICHMENT
────────────────────────────────────────────────────────────
  ID                                 Section                                  total  types
  ─────

In [11]:
for p in sample_parents:
    print(f"\nParent: {p.metadata.get('section_title', '')[:60]}")
    for i, child in enumerate(p.children):
        if child.chunk_type not in (ChunkType.IMAGE, ChunkType.TABLE):
            continue
        if not child.retrieved_content:
            continue
        print(f"\n  [{i}] {child.chunk_type.value}")
        print(f"  retrieved : {child.retrieved_content}")
        print(f"  embedded  : {child.embedding_content}")
        print(f"  {'─'*80}")


Parent: 2.2 Service Level Agreements (SLA)

  [1] image
  retrieved : The diagram illustrates a multi-layered architecture for handling client requests. It begins with "Client Requests" being directed to "Page Rendering Components." These components route requests through a "Request Routing" layer to "Aggregator Services." Another "Request Routing" layer directs requests to various data storage solutions, including "Dynamo instances," "Amazon S3," and "Other datastores." The diagram represents a scalable and distributed system for processing and storing data efficiently.
  embedded  : Section: 2.2 Service Level Agreements (SLA)
Type: image
Content: The diagram illustrates a multi-layered architecture for handling client requests. It begins with "Client Requests" being directed to "Page Rendering Components." These components route requests through a "Request Routing" layer to "Aggregator Services." Another "Request Routing" layer directs requests to various data storage solutions, inc

## 4. Test: Embedder
Embed a list of texts and inspect the vector shapes.

In [12]:
from rag.embedders.openai_embedder import OpenAIEmbedder

embedder = OpenAIEmbedder()   # default: text-embedding-3-small

test_texts = [
    "Revenue grew 15% year-over-year.",
    "Operating expenses increased by 8%.",
    "Net profit margin improved to 22%.",
]

vectors = embedder.embed_documents(test_texts)
print(f"Embedded {len(vectors)} texts")
print(f"Vector dimension: {len(vectors[0])}")
print(f"First vector (first 8 dims): {vectors[0][:8]}")

query_vec = embedder.embed_query("What was the revenue growth?")
print(f"\nQuery vector dim: {len(query_vec)}")

Embedded 3 texts
Vector dimension: 1536
First vector (first 8 dims): [-0.005184173583984375, 0.03564453125, 0.0010614395141601562, 0.042877197265625, 0.0304412841796875, 0.022918701171875, -0.0594482421875, 0.083984375]

Query vector dim: 1536


## 5. Test: Vector Store (Chroma)
Upsert documents and run a similarity search.

In [ ]:
from langchain_core.documents import Document
from rag.vectorstores.chroma_store import ChromaVectorStore
from rag.vectorstores.base import SearchParams

vs = ChromaVectorStore(
    collection_name="rag_test",
    persist_directory="./chroma_test_db",
)

# Build a tiny synthetic dataset
docs = [
    Document(page_content=t, metadata={"source": "test", "chunk_type": "text", "chunk_id": t})
    for t in test_texts
]
vecs = embedder.embed_documents([d.page_content for d in docs])

vs.upsert(docs, vecs)
print("Upserted", len(docs), "documents")

results = vs.search(query_vector=query_vec, params=SearchParams(top_k=2))
print(f"\nSearch results ({len(results)} chunks):")
for r in results:
    print(" -", r.page_content)

## 6. Test: Full Pipeline (end-to-end)
Run the complete RAG pipeline on a file.

In [ ]:
from rag.hybrid.pipeline import HybridRAGPipeline
from rag.vectorstores.chroma_store import ChromaVectorStore
from rag.vectorstores.base import SearchParams

pipeline = HybridRAGPipeline(
    vectorstore=ChromaVectorStore(
        collection_name="rag_e2e_test",
        persist_directory="./chroma_e2e_db",
    )
)

# ── Ingest ──
parent_chunks = pipeline.run(
    file_path=FILE_PATH,
    parser=PARSER,
)
print(f"Ingested {len(parent_chunks)} parent sections")

In [ ]:
# ── Hybrid search ──
QUERY = "your query here"   # ← change this

results = pipeline.search(QUERY, params=SearchParams(top_k=5, use_hybrid=True))

print(f"Top {len(results)} results for: '{QUERY}'\n")
for i, doc in enumerate(results):
    print(f"[{i+1}] score={doc.metadata.get('score')}  type={doc.metadata.get('chunk_type')}  source={doc.metadata.get('source')}")
    print(f"     {doc.page_content[:300]}")
    print()

In [ ]:
# ── Filtered search ──
results_filtered = pipeline.search(
    QUERY,
    params=SearchParams(top_k=5, metadata_filters={"chunk_type": "table"}),
)

print(f"Filtered results (tables only): {len(results_filtered)}")
for doc in results_filtered:
    print(" -", doc.page_content[:200])

## 7. Test: Fusion RAG Pipeline

In [ ]:
from rag.fusion.pipeline import FusionRAGPipeline

fusion = FusionRAGPipeline(pipeline, n_queries=3)

# ── Fusion search ──
results = fusion.search(QUERY, params=SearchParams(top_k=5))

print(f"Top {len(results)} fusion results for: '{QUERY}'\n")
for i, doc in enumerate(results):
    print(f"[{i+1}] rrf_score={doc.metadata.get('rrf_score')}  type={doc.metadata.get('chunk_type')}  source={doc.metadata.get('source')}")
    print(f"     {doc.page_content[:300]}")
    print()

## 8. Test: Evaluator

In [ ]:
from rag.evaluators.llm_evaluator import LLMEvaluator

evaluator = LLMEvaluator(pipeline.enricher.llm)  # reuses pipeline's LLM client

# ── pick your searcher ──
searcher = pipeline   # HybridRAGPipeline
# searcher = fusion   # FusionRAGPipeline

chunks = searcher.search(QUERY, params=SearchParams(top_k=5, use_hybrid=True))
answer = pipeline.generate_answer(QUERY, chunks)
result = evaluator.evaluate_sync(QUERY, chunks, answer)
result.print_summary()

## 9. Test: Multi-Turn / Conversational RAG

When follow-up questions reference prior context (e.g. "What are its limitations?"), the `QueryContextualizer` rewrites them into fully self-contained queries before retrieval.

In [ ]:
from rag.conversation.history import ConversationHistory
from rag.conversation.contextualizer import QueryContextualizer

history = ConversationHistory()

# Simulate a prior turn
history.add(
    question="How does Dynamo handle write conflicts?",
    answer="Dynamo uses vector clocks to track object versions and resolves conflicts during reads using application-defined reconciliation logic.",
)

# Follow-up that only makes sense with context
follow_up = "What are its limitations?"

contextualizer = QueryContextualizer(pipeline.enricher.llm)
standalone = await contextualizer.contextualize(follow_up, history)

print(f"Original : {follow_up}")
print(f"Rewritten: {standalone}")

In [ ]:
# ── Full multi-turn search loop ──
history = ConversationHistory()

questions = [
    "How does Dynamo handle write conflicts?",
    "What are its limitations?",          # ambiguous follow-up
    "How does hinted handoff solve that?", # chains on previous answer
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"  Q: {question}")
    print(f"{'='*60}")

    chunks = pipeline.search(question, params=SearchParams(top_k=3, use_hybrid=True), history=history)
    answer = pipeline.generate_answer(question, chunks)

    print(f"\n  A: {answer}\n")
    history.add(question, answer)